在 Python 多进程编程中，由于每个进程拥有独立的内存空间，进程之间无法直接共享变量。
- mp.Queue（队列）和 mp.Pipe（管道）就是 multiprocessing 模块提供的两种最核心的进程间通信（IPC）机制。

你可以把它们通俗地理解为：
- mp.Queue（队列）：就像一个公共的“公告栏”或“传送带”。<font color='red'>多个进程都可以往上面放东西（生产者），也可以从上面取东西（消费者），非常灵活。</font>
- mp.Pipe（管道）：就像一根私密的“电话线”。<font color='red'>它只有两个端点，通常用于两个进程之间建立一对一的专属通信通道。</font>

下面我为你分别举例并详细解释它们的用法：
## 📦 mp.Queue()：多进程共享的“传送带”
- mp.Queue 是线程和进程安全的，它自带锁机制，因此可以放心地让多个进程同时读写。它遵循先进先出（FIFO）的原则。
适用场景：<font color='red'>多个进程需要同时交换数据，例如经典的“多生产者-多消费者”模型。</font>

In [ ]:
import torch.multiprocessing as mp
import time
import random

def producer(queue, name):
    """生产者：负责往队列里放数据"""
    for i in range(3):
        time.sleep(random.uniform(0.1, 0.5))  # 模拟耗时操作
        item = f"{name} 生产了数据 {i}"
        queue.put(item)  # 将数据放入队列
        print(f"🟢 {item}")

def consumer(queue):
    """消费者：负责从队列里取数据"""
    while True:
        item = queue.get()  # 从队列获取数据（如果队列为空会阻塞等待）
        if item is None:  # 收到 None 信号代表结束
            break
        print(f"🔴 消费者处理了: {item}")
        time.sleep(0.5)

if __name__ == "__main__":
    mp.set_start_method('spawn')
    
    # 创建一个共享队列
    q = mp.Queue()

    # 启动 2 个生产者进程和 1 个消费者进程
    p1 = mp.Process(target=producer, args=(q, "生产者A"))
    p2 = mp.Process(target=producer, args=(q, "生产者B"))
    c = mp.Process(target=consumer, args=(q,))

    p1.start()
    p2.start()
    c.start()

    # 等待生产者完成所有任务
    p1.join()
    p2.join()

    # 生产者都干完活了，给消费者发送结束信号（None）
    q.put(None)
    c.join()

    print("✅ 所有任务处理完毕！")

## 📞 mp.Pipe()：一对一的“专属电话线”
- mp.Pipe() 会返回两个连接对象（通常命名为 conn1, conn2），代表管道的两端。默认情况下它是双向的（duplex=True），两端都可以互相发送（send）和接收（recv）数据。

适用场景：<font color='red'>仅有两个进程需要进行高效、直接的数据传输。</font>

In [ ]:
import torch.multiprocessing as mp
import time

def child_process(conn):
    """子进程逻辑"""
    # 子进程通过管道接收父进程的消息
    msg_from_parent = conn.recv()
    print(f"👶 子进程收到父进程的消息: {msg_from_parent}")
    
    time.sleep(1)
    # 子进程通过管道回复父进程
    conn.send("爸爸，我收到啦！")
    conn.close()

if __name__ == "__main__":
    mp.set_start_method('spawn')
    
    # 创建管道，返回两端的连接对象
    parent_conn, child_conn = mp.Pipe()

    # 将管道的一端（child_conn）传递给子进程
    p = mp.Process(target=child_process, args=(child_conn,))
    p.start()

    # 父进程保留另一端（parent_conn），向子进程发消息
    parent_conn.send("儿子，听到请回答！")
    
    # 父进程接收子进程的回复
    msg_from_child = parent_conn.recv()
    print(f"👨 父进程收到子进程的回复: {msg_from_child}")

    parent_conn.close()
    p.join()

| 特性 | mp.Queue (队列) | mp.Pipe (管道) |
| :--- | :--- | :--- |
| 通信端点 | 可以被多个生产者和消费者共享 | 只有两个端点，仅限两个进程 |
| 安全性 | 自带锁，多进程并发读写非常安全 | 两个端点各自是安全的，但不能让多个进程同时操作同一端 |
| 性能 | 内部基于 Pipe 封装，开销稍大，速度相对较慢 | 更底层、更轻量，两点之间传输速度更快 |
| 易用性 | API 简单（`put` / `get`），不易出错 | 需要手动管理两端的 `send` / `recv`，容易因逻辑不当导致死锁 |

## 总结建议：
- 在绝大多数日常的多进程任务（如深度学习的数据加载、任务分发）中，优先推荐使用 mp.Queue。它的容错率高，编程模型更简单。
- 只有当你明确知道只有两个进程需要频繁、高速地交换数据，并且对性能有极致要求时，才考虑使用 mp.Pipe